In [1]:
platformID = 'TWI'

In [2]:
from datetime import datetime
import pandas as pd

import psycopg2

import os

## import helper

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader, execute_sql_query, compare_or_update_reference
import test_functions

In [4]:
lookup = lookup_loader(gam_info, platformID, '1e')

week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']


✅ Test TWI_1e_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TWI_1e_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TWI_1e_02 passed: No empty values in lookup.
...updating logbook...

✅ Test TWI_1e_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TWI_1e_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TWI_1e_05 passed: No empty values in lookup.
...updating logbook...



# ingestion

## activity

In [5]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

sql_query = f"""
    SELECT 
        week_commencing,
        account_id,
        SUM(engagements) AS tweet_engagements,
        SUM(video_views) AS video_video_views
    FROM
        redshiftdb.central_insights.adverity_social_x_by_tweets
    WHERE
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    GROUP BY
        week_commencing,
        account_id
    ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_activity_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['account_id'] = platformID+ df['account_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

twitter_activity_raw = pd.read_csv(file, keep_default_na=False)

if not twitter_activity_raw['account_id'].str.startswith(platformID, na=False).all():
    twitter_activity_raw['account_id'] = platformID + twitter_activity_raw['account_id']

twitter_activity_raw['account_id'] = twitter_activity_raw['account_id']
twitter_activity_raw['week_commencing'] = pd.to_datetime(twitter_activity_raw['week_commencing'])
twitter_activity = twitter_activity_raw.rename(columns={'account_id': 'Channel ID',
                                                            'week_commencing': 'w/c'})

# keep only measured accounts
print(twitter_activity.shape)
twitter_activity = twitter_activity.merge(socialmedia_accounts[['Channel ID', 'ServiceID']], 
                                          on='Channel ID', how='right')
print(twitter_activity.shape)

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(twitter_activity, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1e_06",
                                             "Check that all page IDs are found in SQL")

# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=twitter_activity,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts,
                                               test_number=f"{platformID}_1e_07",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(twitter_activity, 
                           numeric_columns=['tweet_engagements', 'video_video_views'], 
                           test_number=f"{platformID}_1e_08",
                           test_step='Check no missing values in metric columns column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(twitter_activity, ['Channel ID', 'w/c'], 
                               test_number=f"{platformID}_1e_09",
                               test_step='Check no duplicates from redshift returned')


no redshift connection using last time queried
(2934, 4)
(758, 5)
...testing Channel ID...
✅ Test TWI_1e_06 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
       Channel ID      Start End        w/c
132   TWI18164632 2020-04-01 NaT 2025-03-31
792     TWI790728 2020-04-01 NaT 2025-03-31
264   TWI18308305 2020-04-01 NaT 2025-03-31
836     TWI791197 2020-04-01 NaT 2025-03-31
176   TWI18168536 2020-04-01 NaT 2025-03-31
..            ...        ...  ..        ...
569  TWI372754427 2020-04-01 NaT 2026-01-12
570  TWI372754427 2020-04-01 NaT 2026-01-19
394   TWI23763445 2020-04-01 NaT 2026-01-19
571  TWI372754427 2020-04-01 NaT 2026-01-26
395   TWI23763445 2020-04-01 NaT 2026-01-26

[122 rows x 4 columns]

Summary of missing weeks per channel_id_col:
     Channel ID  missing_week_count
3   TWI23763445                  43
5  TWI372754427                  42
4   TWI36670025                  11
6     TWI790728                   8
1   TWI18168536              

In [6]:


file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

twitter_activity.to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT.csv",
                        index=None)
'''
compare_or_update_reference(twitter_activity[cols], 
                            f"../test/refactoring/{platformID}_expected.pkl", 
                            cols, update=False)
'''

'\ncompare_or_update_reference(twitter_activity[cols], \n                            f"../test/refactoring/{platformID}_expected.pkl", \n                            cols, update=False)\n'